# Naloxone related posts on Reddit

This notebook reuses the subreddit archive from `reddit_user_representativeness.ipynb` to measure how often naloxone (also known as Narcan) appears in the selected communities and to explore how people talk about it.

In [1]:
import json
import os
import re
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm
import zstandard

pd.set_option('display.max_columns', 20)


In [2]:
subreddits = ['addiction', 'fentanyl', 'Drugs', 'FentanylRecovery', 'opiates', 'OpiatesRecovery']
input_dir = Path('../reddit/subreddits24')
start_time = datetime(2014, 1, 1, tzinfo=timezone.utc).timestamp()

keyword_terms = [
    'naloxone',
    'narcan',
]
escaped_keywords = sorted((re.escape(term) for term in keyword_terms), key=len, reverse=True)
keyword_pattern = re.compile(r"\b(?:" + "|".join(escaped_keywords) + r")\b", re.IGNORECASE)

analysis_timezone = timezone.utc

input_dir.exists()


True

In [3]:
def read_and_decode(reader, chunk_size, max_window_size, previous_chunk=None, bytes_read=0):
    chunk = reader.read(chunk_size)
    bytes_read += chunk_size
    if previous_chunk is not None:
        chunk = previous_chunk + chunk
    try:
        return chunk.decode()
    except UnicodeDecodeError:
        if bytes_read > max_window_size:
            raise UnicodeError(f'Unable to decode frame after reading {bytes_read:,} bytes')
        return read_and_decode(reader, chunk_size, max_window_size, chunk, bytes_read)


def read_lines_zst(file_name):
    with open(file_name, 'rb') as file_handle:
        buffer = ''
        reader = zstandard.ZstdDecompressor(max_window_size=2**31).stream_reader(file_handle)
        while True:
            chunk = read_and_decode(reader, 2**27, (2**29) * 2)
            if not chunk:
                break
            lines = (buffer + chunk).split('\n')
            for line in lines[:-1]:
                yield line, file_handle.tell()
            buffer = lines[-1]
        reader.close()

def extract_text(obj, kind):
    if kind == 'submissions':
        title = obj.get('title') or ''
        selftext = obj.get('selftext') or ''
        if selftext in {'[removed]', '[deleted]'}:
            selftext = ''
        text = f"{title}\n{selftext}".strip()
    else:
        body = obj.get('body') or ''
        if body in {'[removed]', '[deleted]'}:
            body = ''
        text = body
    return text

def clean_text(text):
    return re.sub(r'\s+', ' ', text or '').strip()

def month_bucket(ts):
    return datetime.fromtimestamp(ts, tz=analysis_timezone).strftime('%Y-%m')



In [4]:
mention_records = []

for subreddit in subreddits:
    for kind in ('submissions', 'comments'):
        file_path = input_dir / f'{subreddit}_{kind}.zst'
        if not file_path.exists():
            print(f'Missing file: {file_path}')
            continue
        for line, _ in tqdm(read_lines_zst(str(file_path)), desc=f'{subreddit}-{kind}', unit='rows'):
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            created_utc = obj.get('created_utc')
            if created_utc is None:
                continue
            created_ts = int(created_utc)
            if created_ts < start_time:
                continue

            text = extract_text(obj, kind)
            if not text:
                continue

            matches = keyword_pattern.findall(text)
            if not matches:
                continue

            record = {
                'subreddit': subreddit,
                'kind': kind,
                'created_utc': created_ts,
                'created_dt': datetime.fromtimestamp(created_ts, tz=analysis_timezone),
                'month': month_bucket(created_ts),
                'mention_count': len(matches),
                'keyword_hits': sorted({m.lower() for m in matches}),
                'score': obj.get('score'),
                'author': obj.get('author'),
                'id': obj.get('id'),
                'text': clean_text(text),
            }
            if kind == 'submissions':
                record['title'] = obj.get('title') or ''
                record['num_comments'] = obj.get('num_comments')
                record['permalink'] = obj.get('permalink')
            else:
                record['parent_id'] = obj.get('parent_id')
                record['link_id'] = obj.get('link_id')
            mention_records.append(record)

print(f'Total naloxone mentions collected: {len(mention_records):,}')


addiction-submissions: 81516rows [00:02, 27910.07rows/s]
addiction-comments: 531592rows [00:08, 60142.57rows/s]
fentanyl-submissions: 26650rows [00:00, 39059.17rows/s]
fentanyl-comments: 284754rows [00:04, 62880.33rows/s]
Drugs-submissions: 1099233rows [00:22, 48671.90rows/s]
Drugs-comments: 12692928rows [02:18, 91610.68rows/s]
FentanylRecovery-submissions: 3229rows [00:00, 27912.01rows/s]
FentanylRecovery-comments: 32434rows [00:00, 54431.14rows/s]
opiates-submissions: 368904rows [00:07, 47435.95rows/s]
opiates-comments: 5184595rows [00:55, 93694.79rows/s] 
OpiatesRecovery-submissions: 62299rows [00:01, 31912.16rows/s]
OpiatesRecovery-comments: 787052rows [00:11, 70314.29rows/s] 

Total naloxone mentions collected: 63,898


In [5]:
if mention_records:
    mentions_df = pd.DataFrame(mention_records)
    mentions_df['created_dt'] = pd.to_datetime(mentions_df['created_dt'])
else:
    mentions_df = pd.DataFrame(
        columns=['subreddit', 'kind', 'created_utc', 'created_dt', 'month', 'mention_count', 'keyword_hits', 'score', 'author', 'id', 'text']
    )

# 保存到文件
output_file = 'naloxone_mentions.csv'
mentions_df.to_csv(output_file, index=False)
print(f'Saved {len(mentions_df):,} mentions to {output_file}')

# 显示前几条
display(mentions_df.head())


Saved 63,898 mentions to naloxone_mentions.csv


,subreddit,kind,created_utc,created_dt,month,mention_count,keyword_hits,score,author,id,text,title,num_comments,permalink,parent_id,link_id
0,addiction,submissions,1390859475,2014-01-27 21:51:15+00:00,2014-01,3,[narcan],1,SimplyJordanBlog,1wbcj0,I almost died yesterday and it's eating at me....,I almost died yesterday and it's eating at me.,0.0,/r/addiction/comments/1wbcj0/i_almost_died_yes...,NaN,NaN
1,addiction,submissions,1402703998,2014-06-13 23:59:58+00:00,2014-06,2,[naloxone],4,[deleted],283fhs,I need help getting into an outpatient drug re...,I need help getting into an outpatient drug re...,5.0,/r/addiction/comments/283fhs/i_need_help_getti...,NaN,NaN
2,addiction,submissions,1426024535,2015-03-10 21:55:35+00:00,2015-03,1,[narcan],0,Weekend_Lover,2ylw5t,Can you bring narcan on a plane? Traveling in ...,Can you bring narcan on a plane?,2.0,/r/addiction/comments/2ylw5t/can_you_bring_nar...,NaN,NaN
3,addiction,submissions,1428523577,2015-04-08 20:06:17+00:00,2015-04,1,[naloxone],0,[deleted],31ww21,"Reddit, please help me raise funds for an even...","Reddit, please help me raise funds for an even...",0.0,/r/addiction/comments/31ww21/reddit_please_hel...,NaN,NaN
4,addiction,submissions,1433185626,2015-06-01 19:07:06+00:00,2015-06,1,[naloxone],3,gwydionspen,384846,False Recovery? I have been married to a woman...,False Recovery?,13.0,/r/addiction/comments/384846/false_recovery/,NaN,NaN


In [6]:
# 在notebook中运行
print("数据质量分析：")
print(f"总数: {len(mentions_df):,}")
print(f"\n按长度分布:")
print(mentions_df['text'].str.len().describe())
print(f"\n按分数分布:")
print(mentions_df['score'].describe())
print(f"\n按subreddit分布:")
print(mentions_df['subreddit'].value_counts())

数据质量分析：
总数: 63,898

按长度分布:
count    63898.000000
mean       660.288194
std       1146.664913
min          6.000000
25%        164.000000
50%        342.000000
75%        722.000000
max      39922.000000
Name: text, dtype: float64

按分数分布:
count    63898.000000
mean         6.510689
std         54.088524
min        -70.000000
25%          1.000000
50%          2.000000
75%          3.000000
max       3366.000000
Name: score, dtype: float64

按subreddit分布:
subreddit
opiates             38360
Drugs               15925
OpiatesRecovery      3996
fentanyl             3716
addiction            1422
FentanylRecovery      479
Name: count, dtype: int64
